# National coastal GHG Monte Carlo analysis: GWP100, GWP20, and carbon-burial offsets




In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------
# File settings
# -----------------------------
INPUT_MERGED = Path("National_C_GHG_merged.xlsx")
ARCHIVED_GWP100 = Path("National_GHG_component_offset_MC.xlsx")
OPTIONAL_VERIFY = Path("GWP20_sensitivity_supplementary_tables.xlsx")

OUTPUT_XLSX = Path("Final_GHG_GWP100_GWP20_MC_results.xlsx")

# -----------------------------
# Monte Carlo settings
# -----------------------------
N_SIM = 10_000
RANDOM_SEED = 20260913

# -----------------------------
# GWP coefficients
# -----------------------------
CH4_GWP100 = 27.0
CH4_GWP20 = 79.7
N2O_GWP100 = 273.0
N2O_GWP20 = 273.0
CO2_GWP = 1.0

CH4_REWEIGHT = CH4_GWP20 / CH4_GWP100

print(f"CH4 GWP20/GWP100 rescaling factor = {CH4_REWEIGHT:.9f}")

## 1. Load and prepare province × ecosystem inputs


In [ ]:
if not INPUT_MERGED.exists():
    raise FileNotFoundError(f"Missing required input: {INPUT_MERGED}")

df = pd.read_excel(INPUT_MERGED, sheet_name="Merged_Province_Ecotype")
df = df.dropna(subset=["Ecosystem", "Province"]).copy()

province_fix = {
    "Jiansu": "Jiangsu",
    "Hongkong": "Hong Kong",
    "Hong Kong": "Hong Kong",
}

def clean_province(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return province_fix.get(x, x)

def assign_region(province):
    north = {"Liaoning", "Hebei", "Tianjin", "Tianjin and Hebei", "Shandong"}
    east = {"Jiangsu", "Shanghai", "Zhejiang", "Fujian"}
    south = {"Guangdong", "Guangxi", "Hainan", "Hong Kong", "Macao", "Taiwan"}
    if province in north:
        return "North"
    if province in east:
        return "East"
    if province in south:
        return "South"
    return "Other"

def parse_numeric(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    x = str(x).strip()
    if x.startswith("<"):
        try:
            return float(x.replace("<", "")) / 2
        except ValueError:
            return np.nan
    try:
        return float(x)
    except ValueError:
        return np.nan

df["Province_clean"] = df["Province"].apply(clean_province)
df["Region"] = df["Province_clean"].apply(assign_region)

for col in ["Area low (ha)", "Area high (ha)", "Area single (ha)"]:
    df[col + "_numeric"] = df[col].apply(parse_numeric)

def get_area_info(row):
    single = row["Area single (ha)_numeric"]
    low = row["Area low (ha)_numeric"]
    high = row["Area high (ha)_numeric"]

    if pd.notna(single):
        return pd.Series({
            "Area_min_ha": single,
            "Area_max_ha": single,
            "Area_point_ha": single
        })

    candidates = [x for x in [low, high] if pd.notna(x)]
    if len(candidates) == 2:
        amin, amax = min(candidates), max(candidates)
        return pd.Series({
            "Area_min_ha": amin,
            "Area_max_ha": amax,
            "Area_point_ha": (amin + amax) / 2
        })
    if len(candidates) == 1:
        area = candidates[0]
        return pd.Series({
            "Area_min_ha": area,
            "Area_max_ha": area,
            "Area_point_ha": area
        })

    return pd.Series({
        "Area_min_ha": np.nan,
        "Area_max_ha": np.nan,
        "Area_point_ha": np.nan
    })

df = pd.concat([df, df.apply(get_area_info, axis=1)], axis=1)

component_columns = {
    "CO2": "CO2_CO2eq_mean",
    "CH4": "CH4_CO2eq_mean",
    "N2O": "N2O_CO2eq_mean",
}

for col in component_columns.values():
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[["Ecosystem", "Province_clean", "Region", "Area_min_ha", "Area_max_ha"]].head())

## 2. Build gas-specific donor pools



In [ ]:
regional_pools = {}
national_pools = {}

for gas, col in component_columns.items():
    regional_pools[gas] = {}
    national_pools[gas] = {}

    for ecosystem in df["Ecosystem"].unique():
        vals = df.loc[
            (df["Ecosystem"] == ecosystem) & df[col].notna(),
            col
        ].dropna().values
        national_pools[gas][ecosystem] = vals.tolist()

        for region in df["Region"].unique():
            rvals = df.loc[
                (df["Ecosystem"] == ecosystem)
                & (df["Region"] == region)
                & df[col].notna(),
                col
            ].dropna().values
            regional_pools[gas][(ecosystem, region)] = rvals.tolist()

def get_donor_pool(row, gas):
    ecosystem = row["Ecosystem"]
    region = row["Region"]
    col = component_columns[gas]
    measured = row[col]

    if pd.notna(measured):
        return [measured], "Measured"

    regional_pool = regional_pools[gas].get((ecosystem, region), [])
    national_pool = national_pools[gas].get(ecosystem, [])

    if len(regional_pool) >= 2:
        return regional_pool, f"Regional donor ({region})"
    if len(national_pool) >= 1:
        return national_pool, "National ecosystem donor"
    return [], "No donor"

for gas in ["CO2", "CH4", "N2O"]:
    pools, methods, donor_n = [], [], []
    for _, row in df.iterrows():
        pool, method = get_donor_pool(row, gas)
        pools.append(pool)
        methods.append(method)
        donor_n.append(len(pool))
    df[f"{gas}_donor_pool"] = pools
    df[f"{gas}_donor_method"] = methods
    df[f"{gas}_donor_n"] = donor_n

## 3. Carbon-burial uncertainty



In [ ]:
burial_info = {
    "Mangrove": {"low": 50.17, "high": 50.17},
    "Salt marsh": {"low": 213.0, "high": 498.0},
    "Seagrass meadow": {"low": 10.47, "high": 10.47},
    "Tidal flat": {"low": 280.0, "high": 1499.0},
}

## 4. Reconstruct ecosystem-level Monte Carlo draws



In [ ]:
def run_component_mc(data, n_sim=N_SIM, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    ecosystems = sorted(data["Ecosystem"].unique())
    results = []

    for sim in range(1, n_sim + 1):
        eco = {
            ecosystem: {"CO2": 0.0, "CH4": 0.0, "N2O": 0.0}
            for ecosystem in ecosystems
        }

        # Province × ecosystem upscaling
        for _, row in data.iterrows():
            amin, amax = row["Area_min_ha"], row["Area_max_ha"]
            if pd.isna(amin):
                continue

            area = amin if np.isclose(amin, amax) else rng.uniform(amin, amax)

            for gas in ["CO2", "CH4", "N2O"]:
                col = component_columns[gas]
                measured = row[col]

                if pd.notna(measured):
                    flux = measured
                else:
                    pool = row[f"{gas}_donor_pool"]
                    if len(pool) == 0:
                        continue
                    flux = rng.choice(pool)

                # g CO2-eq m-2 yr-1 × ha × 1e-5 = Gg CO2-eq yr-1
                eco[row["Ecosystem"]][gas] += flux * area * 1e-5

        # Carbon-burial draw
        burial = {}
        for ecosystem in ecosystems:
            low = burial_info[ecosystem]["low"]
            high = burial_info[ecosystem]["high"]
            burial_C = low if np.isclose(low, high) else rng.uniform(low, high)
            burial[ecosystem] = burial_C * 44.0 / 12.0

        rec = {"Simulation": sim}

        national = {"CO2": 0.0, "CH4": 0.0, "N2O": 0.0, "Burial": 0.0}

        for ecosystem in ecosystems:
            CO2 = eco[ecosystem]["CO2"]
            CH4 = eco[ecosystem]["CH4"]
            N2O = eco[ecosystem]["N2O"]
            B = burial[ecosystem]

            rec[f"{ecosystem}_CO2_GWP100"] = CO2
            rec[f"{ecosystem}_CH4_GWP100"] = CH4
            rec[f"{ecosystem}_N2O_GWP100"] = N2O
            rec[f"{ecosystem}_Burial"] = B

            # Paired GWP20 reweighting of CH4
            CH4_20 = CH4 * CH4_REWEIGHT
            N2O_20 = N2O

            rec[f"{ecosystem}_CH4_GWP20"] = CH4_20
            rec[f"{ecosystem}_N2O_GWP20"] = N2O_20

            rec[f"{ecosystem}_NonCO2_GWP100"] = CH4 + N2O
            rec[f"{ecosystem}_NonCO2_GWP20"] = CH4_20 + N2O_20

            rec[f"{ecosystem}_Offset_GWP100"] = (CH4 + N2O) / B * 100
            rec[f"{ecosystem}_Offset_GWP20"] = (CH4_20 + N2O_20) / B * 100

            rec[f"{ecosystem}_N2Oshare_GWP100"] = (
                N2O / (CH4 + N2O) * 100 if (CH4 + N2O) != 0 else np.nan
            )
            rec[f"{ecosystem}_N2Oshare_GWP20"] = (
                N2O_20 / (CH4_20 + N2O_20) * 100 if (CH4_20 + N2O_20) != 0 else np.nan
            )

            national["CO2"] += CO2
            national["CH4"] += CH4
            national["N2O"] += N2O
            national["Burial"] += B

        rec["National_CO2_reconstructed"] = national["CO2"]
        rec["National_CH4_GWP100_reconstructed"] = national["CH4"]
        rec["National_N2O_GWP100_reconstructed"] = national["N2O"]
        rec["National_Burial_reconstructed"] = national["Burial"]

        results.append(rec)

    return pd.DataFrame(results)

eco_mc = run_component_mc(df)
eco_mc.head()

## 5. National GWP20 sensitivity



In [ ]:
if not ARCHIVED_GWP100.exists():
    raise FileNotFoundError(
        f"Missing required archived baseline: {ARCHIVED_GWP100}. "
        "This file is required for exact manuscript-level national reproduction."
    )

national_mc = pd.read_excel(ARCHIVED_GWP100, sheet_name="National_MC_draws").copy()

required = [
    "Carbon_burial_CO2eq_Gg",
    "CO2_Gg",
    "CH4_Gg",
    "N2O_Gg",
]
missing = [c for c in required if c not in national_mc.columns]
if missing:
    raise ValueError(f"Archived national draw sheet is missing columns: {missing}")

national_mc["CH4_GWP100_Gg"] = national_mc["CH4_Gg"]
national_mc["N2O_GWP100_Gg"] = national_mc["N2O_Gg"]
national_mc["NonCO2_GWP100_Gg"] = (
    national_mc["CH4_GWP100_Gg"] + national_mc["N2O_GWP100_Gg"]
)

national_mc["CH4_GWP20_Gg"] = national_mc["CH4_GWP100_Gg"] * CH4_REWEIGHT
national_mc["N2O_GWP20_Gg"] = national_mc["N2O_GWP100_Gg"]
national_mc["NonCO2_GWP20_Gg"] = (
    national_mc["CH4_GWP20_Gg"] + national_mc["N2O_GWP20_Gg"]
)

B = national_mc["Carbon_burial_CO2eq_Gg"]

national_mc["Offset_GWP100_pct"] = national_mc["NonCO2_GWP100_Gg"] / B * 100
national_mc["Offset_GWP20_pct"] = national_mc["NonCO2_GWP20_Gg"] / B * 100

national_mc["N2Oshare_GWP100_pct"] = (
    national_mc["N2O_GWP100_Gg"] / national_mc["NonCO2_GWP100_Gg"] * 100
)
national_mc["N2Oshare_GWP20_pct"] = (
    national_mc["N2O_GWP20_Gg"] / national_mc["NonCO2_GWP20_Gg"] * 100
)
national_mc["Offset_delta_pp"] = (
    national_mc["Offset_GWP20_pct"] - national_mc["Offset_GWP100_pct"]
)

national_mc.head()

## 6. Summaries


In [ ]:
def qsummary(x):
    x = pd.Series(x).dropna()
    return {
        "Mean": x.mean(),
        "Median": x.median(),
        "SD": x.std(),
        "P2.5": x.quantile(0.025),
        "P25": x.quantile(0.25),
        "P75": x.quantile(0.75),
        "P97.5": x.quantile(0.975),
    }

# National manuscript-reproduction summary
national_metrics = {
    "Carbon burial climate benefit": "Carbon_burial_CO2eq_Gg",
    "CO2 water-air": "CO2_Gg",
    "CH4 GWP100": "CH4_GWP100_Gg",
    "N2O GWP100": "N2O_GWP100_Gg",
    "Non-CO2 GWP100": "NonCO2_GWP100_Gg",
    "Non-CO2 offset GWP100": "Offset_GWP100_pct",
    "N2O share GWP100": "N2Oshare_GWP100_pct",
    "CH4 GWP20": "CH4_GWP20_Gg",
    "N2O GWP20": "N2O_GWP20_Gg",
    "Non-CO2 GWP20": "NonCO2_GWP20_Gg",
    "Non-CO2 offset GWP20": "Offset_GWP20_pct",
    "N2O share GWP20": "N2Oshare_GWP20_pct",
    "Offset change GWP20-GWP100": "Offset_delta_pp",
}

rows = []
for metric, col in national_metrics.items():
    unit = "%" if ("offset" in metric.lower() or "share" in metric.lower() or "change" in metric.lower()) else "Gg CO2-eq yr-1"
    rows.append({"Scale": "National", "Metric": metric, "Unit": unit, **qsummary(national_mc[col])})

national_summary = pd.DataFrame(rows)

# Ecosystem GWP20 reconstruction
ecosystem_rows = []
ecosystems = sorted(df["Ecosystem"].unique())

for ecosystem in ecosystems:
    metric_map = {
        "Carbon burial climate benefit": f"{ecosystem}_Burial",
        "CH4 GWP100 reconstructed": f"{ecosystem}_CH4_GWP100",
        "N2O GWP100 reconstructed": f"{ecosystem}_N2O_GWP100",
        "Non-CO2 GWP100 reconstructed": f"{ecosystem}_NonCO2_GWP100",
        "Non-CO2 offset GWP100 reconstructed": f"{ecosystem}_Offset_GWP100",
        "CH4 GWP20": f"{ecosystem}_CH4_GWP20",
        "N2O GWP20": f"{ecosystem}_N2O_GWP20",
        "Non-CO2 GWP20": f"{ecosystem}_NonCO2_GWP20",
        "Non-CO2 offset GWP20": f"{ecosystem}_Offset_GWP20",
        "N2O share GWP20": f"{ecosystem}_N2Oshare_GWP20",
    }
    for metric, col in metric_map.items():
        unit = "%" if ("offset" in metric.lower() or "share" in metric.lower()) else "Gg CO2-eq yr-1"
        ecosystem_rows.append({
            "Scale": ecosystem,
            "Metric": metric,
            "Unit": unit,
            **qsummary(eco_mc[col])
        })

ecosystem_summary = pd.DataFrame(ecosystem_rows)

display(national_summary)
display(ecosystem_summary.head(12))

In [ ]:
archived_summary = pd.read_excel(
    ARCHIVED_GWP100,
    sheet_name="Component_offset_summary"
)

diag_rows = []
for ecosystem in ecosystems:
    for component, metric_name in [
        ("CH4", "CH4"),
        ("N2O", "N2O"),
        ("CH4+N2O", "CH4+N2O"),
    ]:
        original_row = archived_summary[
            (archived_summary["Ecosystem"] == ecosystem)
            & (archived_summary["Metric"] == metric_name)
        ]
        if original_row.empty:
            continue

        original_median = float(original_row.iloc[0]["Median"])

        if component == "CH4":
            reconstructed = eco_mc[f"{ecosystem}_CH4_GWP100"]
        elif component == "N2O":
            reconstructed = eco_mc[f"{ecosystem}_N2O_GWP100"]
        else:
            reconstructed = eco_mc[f"{ecosystem}_NonCO2_GWP100"]

        reconstructed_median = reconstructed.median()

        diag_rows.append({
            "Ecosystem": ecosystem,
            "Component": component,
            "Archived_GWP100_median": original_median,
            "Reconstructed_GWP100_median": reconstructed_median,
            "Difference_pct": (
                (reconstructed_median - original_median) / original_median * 100
                if original_median != 0 else np.nan
            )
        })

diagnostics = pd.DataFrame(diag_rows)
display(diagnostics)

## 7. Export outputs



In [ ]:
donor_rows = []
for _, row in df.iterrows():
    for gas in ["CO2", "CH4", "N2O"]:
        donor_rows.append({
            "Ecosystem": row["Ecosystem"],
            "Province": row["Province"],
            "Province_clean": row["Province_clean"],
            "Region": row["Region"],
            "Gas": gas,
            "Method": row[f"{gas}_donor_method"],
            "Donor_n": row[f"{gas}_donor_n"],
            "Donor_values": ", ".join(f"{x:.6g}" for x in row[f"{gas}_donor_pool"]),
        })

donor_information = pd.DataFrame(donor_rows)

method_notes = pd.DataFrame({
    "Item": [
        "Monte Carlo iterations",
        "Random seed",
        "Area uncertainty",
        "Missing flux rule",
        "Carbon burial",
        "GWP100 CH4",
        "GWP20 CH4",
        "N2O GWP",
        "National GWP20",
        "Ecosystem GWP20",
        "Primary climatic metric",
        "Key limitations",
    ],
    "Description": [
        str(N_SIM),
        str(RANDOM_SEED),
        "Single areas fixed; two estimates sampled Uniform(numerical min, numerical max); '<x' represented as x/2.",
        "Gas-specific same-ecosystem regional donor when >=2 regional donors; otherwise national same-ecosystem donor.",
        "Mangrove 50.17 fixed; salt marsh Uniform(213,498); seagrass meadow 10.47 fixed; tidal flat Uniform(280,1499) Gg C yr-1; convert by 44/12.",
        str(CH4_GWP100),
        str(CH4_GWP20),
        str(N2O_GWP100),
        "Exact paired reweighting of CH4 within archived 10,000 GWP100 national draws; N2O and burial retained.",
        "10,000-draw reconstruction from original province × ecosystem input distributions; CH4 reweighted within each reconstructed draw.",
        "Signed CH4 + N2O CO2-equivalent forcing divided by carbon-burial CO2 equivalent ×100.",
        "Spring/summer-dominated observations, annualization to full year, no tidal inundation weighting, incomplete spatial coverage, donor-based missing values, no station-level bootstrap in component analysis.",
    ],
})

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    national_summary.to_excel(writer, sheet_name="National_summary", index=False)
    ecosystem_summary.to_excel(writer, sheet_name="Ecosystem_summary", index=False)
    national_mc.to_excel(writer, sheet_name="National_paired_draws", index=False)
    eco_mc.to_excel(writer, sheet_name="Ecosystem_reconstruction", index=False)
    donor_information.to_excel(writer, sheet_name="Donor_information", index=False)
    diagnostics.to_excel(writer, sheet_name="Diagnostics", index=False)
    method_notes.to_excel(writer, sheet_name="Method_notes", index=False)

print(f"Saved: {OUTPUT_XLSX.resolve()}")